# Exercise P9.1: Logistic Regression & Train/Test Evaluation
### STAT 540 — Week 9


## Overview

Fit a logistic regression to predict a binary outcome, evaluate on held-out data, and report coefficients as odds ratios.

## Task 1: Fit Logistic Regression

Using the Titanic dataset (or another approved binary classification dataset):

In [1]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

# Load Titanic dataset from a public online URL
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
titanic = pd.read_csv(url)

# Standardize column names to lowercase to match the formula
titanic.columns = titanic.columns.str.lower()

# Drop missing values in the specified columns
titanic = titanic.dropna(subset=["survived", "pclass", "sex", "age", "fare"])

fit = smf.logit("survived ~ pclass + sex + age + fare", data=titanic).fit()
print(fit.summary())

# Odds ratios
odds_ratios = np.exp(fit.params)
print("\nOdds Ratios:")
print(odds_ratios.round(3))

Optimization terminated successfully.
         Current function value: 0.453242
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               survived   No. Observations:                  714
Model:                          Logit   Df Residuals:                      709
Method:                           MLE   Df Model:                            4
Date:                Fri, 06 Nov 2026   Pseudo R-squ.:                  0.3290
Time:                        23:42:53   Log-Likelihood:                -323.61
converged:                       True   LL-Null:                       -482.26
Covariance Type:            nonrobust   LLR p-value:                 2.020e-67
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       4.9880      0.572      8.717      0.000       3.867       6.110
sex[T.male]    -2.5182    

**Your turn:** Interpret the odds ratio for `sex[T.male]` and `pclass` in plain language:

> Holding class, age, and fare constant, the odds of survival for men were about 0.08 times the odds for women, meaning roughly 92% lower odds (95% CI for the OR: e^-2.926 to e^-2.111, about 0.05 to 0.12). Being male was the strongest negative predictor in the model.

## Task 2: Train/Test Split Evaluation

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X = titanic[["pclass", "age", "fare"]].dropna()
y = titanic.loc[X.index, "survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)
print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy:  {test_acc:.3f}")
print(classification_report(y_test, model.predict(X_test)))

Train accuracy: 0.708
Test accuracy:  0.671
              precision    recall  f1-score   support

           0       0.70      0.80      0.75        87
           1       0.60      0.46      0.53        56

    accuracy                           0.67       143
   macro avg       0.65      0.63      0.64       143
weighted avg       0.66      0.67      0.66       143



**Your turn:** Is there a gap between train and test accuracy? What does this tell you about overfitting?

> The test accuracy is slightly lower, but it depends on what order of magnitude one might use to determine whether or not the gap indicates overfitting. Here the gap is 0.037 (0.708 vs 0.671), which is small. With only 143 test observations, the standard error of the test accuracy is about 0.04, so the gap is within about one standard error and could easily be sampling noise. There is little evidence of meaningful overfitting. The bigger concern is that both accuracies are modest: the test accuracy is only about six points above the 0.608 majority-class baseline (87/143), and recall for survivors is just 0.46. That pattern points more toward an underpowered, simple model (underfitting) than toward overfitting.

## Task 3: What This Model Cannot Do

Write a 150-word limitations section for this model:

> This logistic regression is limited in several ways. First, it is associative, not causal: it shows who was more likely to survive, not why. Second, performance is modest. Test accuracy (0.671) is only about six points above the 0.608 majority-class baseline, and recall for survivors is 0.46, so the model misses over half of those who lived. Third, it assumes linear effects on the log-odds with no interactions, so patterns like class-by-sex differences are missed, and treating pclass as numeric assumes equal steps between classes. Fourth, the data are small and specific: 714 passengers with recorded age, a 143-row test set, and a single historical event, so results may not generalize to other populations. Fifth, the default 0.5 threshold favors the majority class (non-survivors), which explains the low survivor recall; a different threshold or class weights could trade precision for recall. Missing-age rows were also dropped, which may bias the sample.

## Submission

```bash
git add week09/exercises/P9.1*
git commit -m "Complete Exercise P9.1: Logistic regression with train/test evaluation"
git push origin main
```